# Workshop: Tunix-Med · Part 3: Supervised Fine-Tuning with JAX & Tunix

This is the core of the workshop. We use **Tunix**, a high-performance JAX-based fine-tuning library, to train the **Gemma 3 1B** model on our cardiology dataset.

### Why JAX and Tunix?
1. **XLA Compilation**: JAX uses the XLA compiler to fuse operations into highly efficient kernels.
2. **Streaming Data**: A pipeline that never loads the whole dataset into RAM.
3. **Advanced LoRA**: Low-Rank Adaptation applied to both Linear *and* Einsum layers in Gemma 3's attention.


## 1 · Environment & JAX Setup

In [1]:
import os
import logging

# Prevent JAX from hogging all VRAM immediately
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Silence verbose library logging for a cleaner workshop experience
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
for _name in ("absl", "jax", "orbax", "flax", "datasets", "transformers", "grain"):
    logging.getLogger(_name).setLevel(logging.ERROR)

import jax
import jax.numpy as jnp
import numpy as np

print(f"JAX backend : {jax.default_backend()}")
print(f"Devices     : {jax.devices()}")

JAX backend : gpu
Devices     : [CudaDevice(id=0)]


## 2 · Hyperparameter Configuration

`MAX_STEPS` is computed **after** the dataset split is created (Cell 4), because it
depends on the number of training examples.  All other hyper-parameters live here.


In [ ]:
class Config:
    MODEL_NAME = "google/gemma-3-270M-it"
    MODEL_KEY = "gemma-3-270M-it"
    TUNED_MODEL_NAME = f"lmassaron/{MODEL_KEY}-medical-cardiology-lora"

    DATASET_PATH = "data/medical-cardiology-qa"
    MAX_SEQ_LEN = 1024
    EVAL_SPLIT = 0.1
    SEED = 42

    # LoRA
    LORA_RANK = 16
    LORA_ALPHA = 16
    LORA_DROPOUT = 0.05

    # Optimisation
    LEARNING_RATE = 5e-5
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1

    # Training loop
    BATCH_SIZE = 16
    GRAD_ACCUM_STEPS = 2
    NUM_EPOCHS = 3
    EARLY_STOP_PATIENCE = 3
    EVAL_EVERY_N_STEPS = 20
    LOG_EVERY_N_STEPS = 20
    OUTPUT_DIR = "tunix-medical-model"

    # Computed below, after the dataset is loaded
    MAX_STEPS: int = 0


params = Config()
print(f"Effective Batch Size: {params.BATCH_SIZE * params.GRAD_ACCUM_STEPS}")

Effective Batch Size: 32


## 3 · Streaming Data Pipeline

In [3]:
import datasets
from datasets import load_dataset
from transformers import AutoTokenizer
from tunix.sft import peft_trainer

hf_tokenizer = AutoTokenizer.from_pretrained(params.MODEL_NAME)
full_ds = load_dataset("lmassaron/medical-cardiology-qa", split="train")
n = len(full_ds)

# Reproducible train / eval split
rng = np.random.default_rng(params.SEED)
all_idx = rng.permutation(n)
cut = int(n * (1.0 - params.EVAL_SPLIT))
train_idx, eval_idx = all_idx[:cut], all_idx[cut:]

# ── Compute MAX_STEPS now that we know the dataset size ──────────────────────
_steps_per_epoch = len(train_idx) // params.BATCH_SIZE
params.MAX_STEPS = (_steps_per_epoch * params.NUM_EPOCHS) // params.GRAD_ACCUM_STEPS
print(f"Dataset    : {n:,} rows  |  Train: {len(train_idx):,}  Eval: {len(eval_idx):,}")
print(
    f"MAX_STEPS  : {params.MAX_STEPS}  (warmup: {int(params.MAX_STEPS * params.WARMUP_RATIO)})"
)


def _tokenise(example):
    messages = example["messages"]
    full_ids = hf_tokenizer.apply_chat_template(messages, tokenize=True)
    prompt_ids = hf_tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True
    )
    # Extract IDs if it's a BatchEncoding
    if hasattr(full_ids, "input_ids"):
        full_ids = full_ids["input_ids"]
    if hasattr(prompt_ids, "input_ids"):
        prompt_ids = prompt_ids["input_ids"]

    prompt_len = len(prompt_ids)
    full_ids = full_ids[: params.MAX_SEQ_LEN]

    mask = np.ones(len(full_ids), dtype=np.int32)
    mask[: min(prompt_len, len(full_ids))] = 0

    pad_len = params.MAX_SEQ_LEN - len(full_ids)
    tokens = np.pad(
        full_ids, (0, pad_len), constant_values=hf_tokenizer.pad_token_id
    ).astype(np.int32)
    mask = np.pad(mask, (0, pad_len), constant_values=0).astype(np.int32)
    return {"input_tokens": tokens, "input_mask": mask}


def make_data_iterator(dataset, indices, batch_size, shuffle=True, infinite=True):
    while True:
        idxs = indices.copy()
        if shuffle:
            np.random.shuffle(idxs)
        for start in range(0, len(idxs) - batch_size + 1, batch_size):
            batch_idx = idxs[start : start + batch_size]
            processed = [_tokenise(dataset[int(i)]) for i in batch_idx]
            tokens = np.stack([p["input_tokens"] for p in processed])
            masks = np.stack([p["input_mask"] for p in processed])
            yield peft_trainer.TrainingInput(
                input_tokens=jnp.array(tokens),
                input_mask=jnp.array(masks),
            )
        if not infinite:
            break


train_iter = make_data_iterator(full_ds, train_idx, params.BATCH_SIZE)
print("Streaming data pipeline configured.")

Dataset    : 10,518 rows  |  Train: 9,466  Eval: 1,052
MAX_STEPS  : 295  (warmup: 29)
Streaming data pipeline configured.


## 4 · Model & Custom LoRA Setup

Gemma 3 uses **Einsum** operations for its attention.  We provide custom LoRA layers to support both Linear and Einsum modules.


In [4]:
from flax import nnx
from tunix.models import automodel
from huggingface_hub import snapshot_download

# Build model on a 1-D mesh (single-GPU or single-TPU)
devices = np.array(jax.devices()).reshape((1, -1))
mesh = jax.sharding.Mesh(devices, axis_names=("tp", "fsdp"))
model_path = snapshot_download(params.MODEL_NAME)
model_config = automodel.call_model_config(params.MODEL_KEY)

with mesh:
    model = automodel.create_model_from_safe_tensors(
        params.MODEL_KEY, model_path, model_config, mesh, dtype=jnp.bfloat16
    )


# ── Custom LoRA wrappers for Linear and Einsum layers ────────────────────────
class LinearLoRALayer(nnx.Module):
    def __init__(
        self, linear_module, rank, alpha, dropout, dtype=jnp.bfloat16, rngs=None
    ):
        self.linear = linear_module
        self.rank = rank
        self.scale = alpha / rank
        self.dropout = nnx.Dropout(dropout, rngs=rngs)

        in_features = linear_module.in_features
        out_features = linear_module.out_features

        self.lora_a = nnx.LoRAParam(
            jax.random.normal(rngs.params(), (in_features, rank), dtype=dtype) * 0.01
        )
        self.lora_b = nnx.LoRAParam(jnp.zeros((rank, out_features), dtype=dtype))

    def __call__(self, x):
        base = self.linear(x)
        lora = (self.dropout(x) @ self.lora_a[...] @ self.lora_b[...]) * self.scale
        return base + lora.astype(x.dtype)


class EinsumLoRALayer(nnx.Module):
    def __init__(
        self, einsum_module, rank, alpha, dropout, dtype=jnp.bfloat16, rngs=None
    ):
        self.einsum = einsum_module
        self.rank = rank
        self.scale = alpha / rank
        self.dropout = nnx.Dropout(dropout, rngs=rngs)

        w = einsum_module.w
        in_dim = int(np.prod(w.shape[:-1]))
        out_dim = w.shape[-1]

        self.lora_a = nnx.LoRAParam(
            jax.random.normal(rngs.params(), (in_dim, rank), dtype=dtype) * 0.01
        )
        self.lora_b = nnx.LoRAParam(jnp.zeros((rank, out_dim), dtype=dtype))

    def __call__(self, x):
        base = self.einsum(x)
        delta_w = (self.lora_a[...] @ self.lora_b[...]).reshape(self.einsum.w.shape)
        return base + self.scale * jnp.einsum(
            self.einsum.einsum_str, self.dropout(x), delta_w.astype(x.dtype)
        )


def _get_child(parent, key):
    try:
        return getattr(parent, str(key))
    except AttributeError:
        return parent[int(key)]


def _set_child(parent, key, value):
    try:
        setattr(parent, str(key), value)
    except (AttributeError, TypeError):
        parent[int(key)] = value


def apply_lora(model, rank, alpha, dropout):
    replaced = 0
    rngs = nnx.Rngs(params=0, dropout=0)
    for path, mod in nnx.graph.iter_graph(model):
        if not path:
            continue
        parent = model
        try:
            for step in path[:-1]:
                parent = _get_child(parent, step)
            attr = path[-1]
        except Exception:
            continue

        if isinstance(mod, nnx.Linear):
            _set_child(
                parent, attr, LinearLoRALayer(mod, rank, alpha, dropout, rngs=rngs)
            )
            replaced += 1
        elif isinstance(mod, nnx.Einsum):
            _set_child(
                parent, attr, EinsumLoRALayer(mod, rank, alpha, dropout, rngs=rngs)
            )
            replaced += 1
    return replaced


with mesh:
    n_lora = apply_lora(model, params.LORA_RANK, params.LORA_ALPHA, params.LORA_DROPOUT)
print(f"Patched {n_lora} layers with LoRA.")

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Patched 54 layers with LoRA.


## 5 · Training Hooks & Export Logic

In [5]:
import optax
import time
from tunix.sft import hooks


def export_adapter(model, cfg):
    import os, json, safetensors.flax as stf
    from flax import nnx

    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

    # 1. Save adapter_config.json
    config = {
        "base_model_name_or_path": cfg.MODEL_NAME,
        "lora_alpha": cfg.LORA_ALPHA,
        "lora_dropout": cfg.LORA_DROPOUT,
        "r": cfg.LORA_RANK,
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        "peft_type": "LORA",
        "task_type": "CAUSAL_LM",
        "bias": "none",
    }
    with open(os.path.join(cfg.OUTPUT_DIR, "adapter_config.json"), "w") as f:
        json.dump(config, f, indent=2)

    # 2. Map and save weights
    lora_weights = {}
    for path, leaf in nnx.graph.iter_graph(model):
        if not isinstance(leaf, (nnx.Param, nnx.LoRAParam)):
            continue
        key = ".".join(str(p) for p in path)
        if "lora" not in key.lower():
            continue
        peft_key = "base_model.model.model." + key
        peft_key = (
            peft_key.replace(".lora_a", ".lora_A.weight")
            .replace(".lora_b", ".lora_B.weight")
            .replace(".value", "")
        )
        lora_weights[peft_key] = leaf[...]

    if lora_weights:
        stf.save_file(
            lora_weights, os.path.join(cfg.OUTPUT_DIR, "adapter_model.safetensors")
        )
        print(f"  Adapter saved → {cfg.OUTPUT_DIR}/adapter_model.safetensors")


class CleanProgressHook(hooks.TrainingHooks):
    def __init__(
        self, log_every, max_steps, eval_idx_ref, model, params, export_fn, patience=5
    ):
        self._log_every = log_every
        self._max_steps = max_steps
        self._eval_idx = eval_idx_ref
        self._model = model
        self._params = params
        self._export_fn = export_fn
        self._patience = patience
        self._best_loss = float("inf")
        self._no_improve = 0
        self._t0 = time.time()
        self._step = 0
        self.train_loss_history = []
        self.eval_loss_history = []

    def on_train_start(self, train_ctx):
        pass

    def on_train_end(self, train_ctx):
        pass

    def on_train_step_start(self, train_ctx):
        pass

    def on_eval_step_start(self, train_ctx):
        pass

    def on_train_step_end(self, train_ctx, train_step, train_loss, step_time):
        self._step = train_step
        self.train_loss_history.append((self._step, float(train_loss)))
        if self._step % self._log_every == 0:
            elapsed = time.time() - self._t0
            print(
                f"  step {self._step:>5}/{self._max_steps}  loss={float(train_loss):.4f}  elapsed={elapsed:.0f}s"
            )

    def on_eval_step_end(self, train_ctx, eval_loss):
        n_batches = max(1, len(self._eval_idx) // self._params.BATCH_SIZE)
        mean_loss = float(eval_loss) / n_batches
        self.eval_loss_history.append((self._step, mean_loss))
        if mean_loss < self._best_loss:
            self._best_loss = mean_loss
            self._no_improve = 0
            self._export_fn(self._model, self._params)
            print(f"  [eval] step {self._step}  loss={mean_loss:.4f} ↓ — Saved!")
        else:
            self._no_improve += 1
            if self._no_improve >= self._patience:
                raise StopIteration


schedule = optax.warmup_cosine_decay_schedule(
    0.0,
    params.LEARNING_RATE,
    int(params.MAX_STEPS * params.WARMUP_RATIO),
    params.MAX_STEPS,
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=params.WEIGHT_DECAY)

hook = CleanProgressHook(
    params.LOG_EVERY_N_STEPS,
    params.MAX_STEPS,
    eval_idx,
    model,
    params,
    export_adapter,
    params.EARLY_STOP_PATIENCE,
)


def gen_model_input(training_input):
    from tunix.sft import utils as sft_utils

    tokens = training_input.input_tokens
    mask = training_input.input_mask

    positions = sft_utils.build_positions_from_mask(mask)
    attention_mask = sft_utils.make_causal_attn_mask(mask)

    return {
        "input_tokens": tokens,
        "input_mask": mask,
        "positions": positions,
        "attention_mask": attention_mask,
    }


trainer = peft_trainer.PeftTrainer(
    model,
    optimizer,
    peft_trainer.TrainingConfig(
        max_steps=params.MAX_STEPS,
        eval_every_n_steps=params.EVAL_EVERY_N_STEPS,
        gradient_accumulation_steps=params.GRAD_ACCUM_STEPS,
    ),
)
trainer.with_training_hooks(hook)
trainer.with_gen_model_input_fn(gen_model_input)
print("Trainer ready.")

Trainer ready.


## 6 · Execute Training

In [6]:
class ReusableEvalIter:
    def __iter__(self):
        return make_data_iterator(
            full_ds, eval_idx, params.BATCH_SIZE, shuffle=False, infinite=False
        )

with mesh:
    try:
        trainer.train(train_iter, ReusableEvalIter())
    except StopIteration:
        print("Early stopping.")
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise

print(f"Training complete.")

  Adapter saved → tunix-medical-model/adapter_model.safetensors
  [eval] step 0  loss=11.5177 ↓ — Saved!


Training:   0%|          | 0/295 [00:00<?, ?step/s]

  step    50/295  loss=5.2877  elapsed=329s
  Adapter saved → tunix-medical-model/adapter_model.safetensors
  [eval] step 99  loss=4.2248 ↓ — Saved!
  step   100/295  loss=4.4312  elapsed=435s
  step   150/295  loss=4.0991  elapsed=518s
  Adapter saved → tunix-medical-model/adapter_model.safetensors
  [eval] step 199  loss=3.7300 ↓ — Saved!
  step   200/295  loss=3.6270  elapsed=624s
  step   250/295  loss=3.4974  elapsed=707s
Training complete.


## 7 · Upload to Hugging Face Hub

In [7]:
from huggingface_hub import HfApi

api = HfApi()
token = os.environ.get("HF_TOKEN")
print(f"Pushing to {params.TUNED_MODEL_NAME} ...")
try:
    api.create_repo(
        repo_id=params.TUNED_MODEL_NAME,
        repo_type="model",
        private=False,
        exist_ok=True,
        token=token,
    )
    api.upload_file(
        path_or_fileobj=os.path.join(params.OUTPUT_DIR, "adapter_model.safetensors"),
        path_in_repo="adapter_model.safetensors",
        repo_id=params.TUNED_MODEL_NAME,
        token=token,
    )
    api.upload_file(
        path_or_fileobj=os.path.join(params.OUTPUT_DIR, "adapter_config.json"),
        path_in_repo="adapter_config.json",
        repo_id=params.TUNED_MODEL_NAME,
        token=token,
    )
    print(f"Successfully pushed to https://huggingface.co/{params.TUNED_MODEL_NAME}")
except Exception as e:
    print(f"Error pushing to Hub: {e}")

Pushing to lmassaron/gemma-3-270M-it-medical-cardiology-lora ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Successfully pushed to https://huggingface.co/lmassaron/gemma-3-270M-it-medical-cardiology-lora
